In [2]:
!pip install torch torchvision torchaudio scikit-learn matplotlib seaborn grad-cam

In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score,roc_curve
import seaborn as sns
import torch

from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.datasets import ImageFolder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

Using: cuda


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
DATA_DIR = "drive/MyDrive/saved_dataset"
OUTPUT_DIR = "./resnet50_results"
train_path = "/content/drive/MyDrive/saved_dataset/train"
val_path = "/content/drive/MyDrive/saved_dataset/val"
test_path = "/content/drive/MyDrive/saved_dataset/test"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [6]:
IMG_SIZE = 224
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.1)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [7]:
train_dataset = ImageFolder(train_path, transform=train_transform)
val_dataset = ImageFolder(val_path, transform=val_test_transform)
test_dataset = ImageFolder(test_path, transform=val_test_transform)

batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f" Loaded: Train={len(train_dataset)}, Val={len(val_dataset)}, Test={len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")

 Loaded: Train=5982, Val=878, Test=879
Classes: ['NORMAL', 'PNEUMONIA']


In [8]:
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
model.fc = nn.Sequential(nn.Dropout(0.5), nn.Linear(model.fc.in_features, 1))
model = model.to(device)
print(" ResNet50 ready for binary pneumonia classification")

 ResNet50 ready for binary pneumonia classification


In [9]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, 'max', factor=0.5, patience=3)

best_val_auc = 0
train_losses, val_losses, val_aucs = [], [], []

In [10]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    train_loss = 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs = model(imgs).squeeze(-1)
        loss = criterion(outputs, labels.squeeze(-1))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)

            outputs = model(imgs).squeeze(-1)

            # validation loss
            loss = criterion(outputs, labels.squeeze(-1))
            val_loss += loss.item()

            # predictions for AUC
            preds = torch.sigmoid(outputs).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(labels.squeeze(-1).cpu().numpy())

    val_loss /= len(val_loader)
    val_losses.append(val_loss)

    val_auc = roc_auc_score(val_labels, val_preds)
    val_aucs.append(val_auc)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}, Val AUC={val_auc:.4f}")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'bestmodel.pth'))
        print(f"Best model saved to {OUTPUT_DIR}/bestmodel.pth")

    scheduler.step(val_auc)

Epoch 1: Train Loss=0.1775, Val Loss=0.0859, Val AUC=0.9971
Best model saved to ./resnet50_results/bestmodel.pth
Epoch 2: Train Loss=0.0935, Val Loss=0.0647, Val AUC=0.9962
Epoch 3: Train Loss=0.0804, Val Loss=0.0770, Val AUC=0.9949
Epoch 4: Train Loss=0.0715, Val Loss=0.0698, Val AUC=0.9966
Epoch 5: Train Loss=0.0616, Val Loss=0.1242, Val AUC=0.9964
Epoch 6: Train Loss=0.0398, Val Loss=0.0658, Val AUC=0.9970
Epoch 7: Train Loss=0.0299, Val Loss=0.0827, Val AUC=0.9975
Best model saved to ./resnet50_results/bestmodel.pth
Epoch 8: Train Loss=0.0277, Val Loss=0.0595, Val AUC=0.9976
Best model saved to ./resnet50_results/bestmodel.pth
Epoch 9: Train Loss=0.0314, Val Loss=0.0858, Val AUC=0.9951
Epoch 10: Train Loss=0.0333, Val Loss=0.0728, Val AUC=0.9975


In [11]:
import csv

csv_file = os.path.join(OUTPUT_DIR, "training_metrics.csv")

with open(csv_file, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_auc"])
    for i in range(num_epochs):
        writer.writerow([i+1, train_losses[i], val_losses[i], val_aucs[i]])

print(f"Metrics saved to {csv_file}")

Metrics saved to ./resnet50_results/training_metrics.csv


In [20]:

model_path = os.path.join(OUTPUT_DIR, 'bestmodel.pth')
model.load_state_dict(torch.load(model_path))
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(device), labels.float().unsqueeze(1).to(device)
        outputs = torch.sigmoid(model(imgs)).cpu().numpy()
        test_preds.extend(outputs)
        test_labels.extend(labels.cpu().numpy())

test_auc = roc_auc_score(test_labels, test_preds)
preds_bin = (np.array(test_preds) > 0.5).astype(int).flatten()

report = classification_report(test_labels, preds_bin, target_names=["Normal", "Pneumonia"])
print(f" FINAL TEST AUC: {test_auc:.4f}")
print(report)

report_path = os.path.join(OUTPUT_DIR, 'classification_report.txt')
with open(report_path, "w") as f:
    f.write("Pneumonia Detection - Final Test Report\n")
    f.write("="*40 + "\n")
    f.write(f"Test AUC: {test_auc:.4f}\n\n")
    f.write(report)

fpr, tpr, _ = roc_curve(test_labels, test_preds)
roc_df = pd.DataFrame({'fpr': fpr, 'tpr': tpr})
roc_df.to_csv(os.path.join(OUTPUT_DIR, 'roc_curve_data.csv'), index=False)

fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(train_losses, label='Train Loss', color='blue')
if 'val_losses' in locals() or 'val_losses' in globals():
    ax1.plot(val_losses, label='Val Loss', color='red')
ax1.set_title('Training & Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(val_aucs, label='Val AUC', color='green')
ax2.set_title('Validation AUC')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('AUC Score')
ax2.legend()

fig1.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=300, bbox_inches='tight')
plt.close(fig1)

fig2, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(confusion_matrix(test_labels, preds_bin), annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Confusion Matrix')
ax.set_ylabel('True Label')
ax.set_xlabel('Predicted Label')
ax.set_xticklabels(["Normal", "Pneumonia"])
ax.set_yticklabels(["Normal", "Pneumonia"])

fig2.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.close(fig2)

 FINAL TEST AUC: 0.9981
              precision    recall  f1-score   support

      Normal       0.93      0.98      0.95       238
   Pneumonia       0.99      0.97      0.98       641

    accuracy                           0.97       879
   macro avg       0.96      0.98      0.97       879
weighted avg       0.98      0.97      0.98       879



In [17]:
import pandas as pd
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(test_labels, test_preds)
roc_auc = auc(fpr, tpr)

df_roc = pd.DataFrame({
    'fpr': fpr,
    'tpr': tpr,
    'thresholds': thresholds
})

roc_csv_path = os.path.join(OUTPUT_DIR, 'roc_curve_data.csv')
df_roc.to_csv(roc_csv_path, index=False)

print(f"ROC curve data (AUC: {roc_auc:.4f}) saved to: {roc_csv_path}")

ROC curve data (AUC: 0.9981) saved to: ./resnet50_results/roc_curve_data.csv


In [18]:
model_path = os.path.join(OUTPUT_DIR, 'bestmodel.pth')
model.load_state_dict(torch.load(model_path))
model.eval()

normal_folder = "/content/drive/MyDrive/saved_dataset/test/NORMAL"
pneumonia_folder = "/content/drive/MyDrive/saved_dataset/test/PNEUMONIA"

# Get 5 random images from each
normal_files = os.listdir(normal_folder)[:5]
pneumonia_files = os.listdir(pneumonia_folder)[:5]

print(" **PREDICTING 10 TEST IMAGES**")
print("=" * 60)

all_normal_probs, all_pneumonia_probs = [], []

# Predict Normal images
print("\n NORMAL IMAGES:")
for i, filename in enumerate(normal_files):
    img_path = os.path.join(normal_folder, filename)
    img = Image.open(img_path).convert('RGB')
    img_tensor = val_test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        prob_pneumonia = torch.sigmoid(model(img_tensor)).item()
        prediction = "NORMAL" if prob_pneumonia < 0.5 else "PNEUMONIA"

    all_normal_probs.append(prob_pneumonia)
    print(f"  {filename}: {prob_pneumonia:.1%} pneumonia → **{prediction}**")

# Predict Pneumonia images
print("\n PNEUMONIA IMAGES:")
for i, filename in enumerate(pneumonia_files):
    img_path = os.path.join(pneumonia_folder, filename)
    img = Image.open(img_path).convert('RGB')
    img_tensor = val_test_transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        prob_pneumonia = torch.sigmoid(model(img_tensor)).item()
        prediction = "PNEUMONIA" if prob_pneumonia > 0.5 else "NORMAL"

    all_pneumonia_probs.append(prob_pneumonia)
    print(f"  {filename}: {prob_pneumonia:.1%} pneumonia → **{prediction}**")

# Visualize 2x5 grid
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for i, filename in enumerate(normal_files):
    img_path = os.path.join(normal_folder, filename)
    img = Image.open(img_path)
    prob = all_normal_probs[i]
    pred = "NORMAL" if prob < 0.5 else "PNEUMONIA"
    axes[0, i].imshow(img, cmap='gray')
    axes[0, i].set_title(f'{filename[:20]}\n{prob:.0%} pneumonia\n{pred}', fontsize=10)
    axes[0, i].axis('off')

for i, filename in enumerate(pneumonia_files):
    img_path = os.path.join(pneumonia_folder, filename)
    img = Image.open(img_path)
    prob = all_pneumonia_probs[i]
    pred = "PNEUMONIA" if prob > 0.5 else "NORMAL"
    axes[1, i].imshow(img, cmap='gray')
    axes[1, i].set_title(f'{filename[:20]}\n{prob:.0%} pneumonia\n{pred}', fontsize=10)
    axes[1, i].axis('off')

plt.suptitle('Model Predictions: Top=Normal, Bottom=Pneumonia', fontsize=16)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'test_predictions_grid.png')
fig.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close(fig)

 **PREDICTING 10 TEST IMAGES**

 NORMAL IMAGES:
  normal_107.png: 0.5% pneumonia → **NORMAL**
  normal_116.png: 0.1% pneumonia → **NORMAL**
  normal_111.png: 0.1% pneumonia → **NORMAL**
  normal_105.png: 0.0% pneumonia → **NORMAL**
  normal_101.png: 21.6% pneumonia → **NORMAL**

 PNEUMONIA IMAGES:
  pneumonia_241.png: 100.0% pneumonia → **PNEUMONIA**
  pneumonia_251.png: 100.0% pneumonia → **PNEUMONIA**
  pneumonia_254.png: 100.0% pneumonia → **PNEUMONIA**
  pneumonia_218.png: 99.6% pneumonia → **PNEUMONIA**
  pneumonia_232.png: 58.7% pneumonia → **PNEUMONIA**


In [19]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import cv2

# 1. Setup Grad-CAM
# ResNet50 uses 'layer4' as the final convolutional block
target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

samples = []
for i in range(5):
    samples.append((os.path.join(normal_folder, normal_files[i]), "Normal"))
    samples.append((os.path.join(pneumonia_folder, pneumonia_files[i]), "Pneumonia"))

# 3. Initialize the plot: 10 rows (one per image), 3 columns (Orig, Mask, Overlay)
fig, axes = plt.subplots(len(samples), 3, figsize=(15, 4 * len(samples)))

print(f"Generating Grad-CAM for {len(samples)} images...")

for i, (img_path, label) in enumerate(samples):
    # Load and preprocess
    img_pil = Image.open(img_path).convert('RGB')
    rgb_img = np.array(img_pil) / 255.0
    rgb_img_resized = cv2.resize(rgb_img, (IMG_SIZE, IMG_SIZE))

    # Transform for the model
    input_tensor = val_test_transform(img_pil).unsqueeze(0).to(device)

    # Generate CAM mask targeting the "Pneumonia" probability
    targets = [ClassifierOutputTarget(0)]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

    # Create heatmap overlay
    visualization = show_cam_on_image(rgb_img_resized, grayscale_cam, use_rgb=True)

    # Plotting column 1: Original
    axes[i, 0].imshow(rgb_img_resized)
    axes[i, 0].set_title(f"Target: {label}")

    # Plotting column 2: Heatmap
    axes[i, 1].imshow(grayscale_cam, cmap='jet')
    axes[i, 1].set_title("Attention Mask")

    # Plotting column 3: Combined
    axes[i, 2].imshow(visualization)
    axes[i, 2].set_title("Grad-CAM Overlay")

    # Clean up axis labels
    for ax in axes[i]:
        ax.axis('off')

# 4. Save to your output directory
plt.tight_layout()
grad_cam_save_path = os.path.join(OUTPUT_DIR, 'grad_cam_10_samples.png')
fig.savefig(grad_cam_save_path, dpi=200, bbox_inches='tight')
plt.close(fig)


Generating Grad-CAM for 10 images...
